# 01. Data Diagnosis

## Goal

원본 스내치 영상에서 `videos.csv` Manifest를 안전하게 생성하거나 불러오고, 다음 분석 단계 전에 기본 데이터 상태를 확인합니다.

## Setup

이 노트북은 프로젝트 루트 또는 `notebooks/` 디렉터리에서 실행할 수 있습니다. 재사용 로직은 `metadata.py`와 `video.py`에서 가져옵니다.

In [ ]:
from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from snatch_analysis.config import B00_VIDEO_ID
from snatch_analysis.metadata import write_video_manifest
from snatch_analysis.video import (
    iter_video_frames,
    read_video_frame,
    read_video_properties,
)


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요.")

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## Steps

### 1. 경로와 재생성 여부 설정

`REBUILD_MANIFEST=False`가 기본값입니다. 기존 CSV가 있으면 그대로 불러오므로 이후 직접 기록할 `has_plate`, `camera_motion` 값이 유지됩니다. 원본 영상이 추가되었거나 자동 메타데이터를 처음부터 다시 만들 때만 `True`로 바꾸세요.

In [ ]:
VIDEO_DIR = Path("data/raw/videos")
MANIFEST_PATH = Path("data/metadata/videos.csv")
REBUILD_MANIFEST = False

### 2. Manifest 생성 또는 불러오기

CSV가 없거나 `REBUILD_MANIFEST=True`이면 `write_video_manifest()`가 영상을 읽어 새 CSV를 만듭니다. 그 외에는 기존 CSV를 불러옵니다.

In [ ]:
if REBUILD_MANIFEST or not MANIFEST_PATH.exists():
    manifest = write_video_manifest(VIDEO_DIR, MANIFEST_PATH)
    manifest_action = "생성"
else:
    manifest = pd.read_csv(MANIFEST_PATH)
    manifest_action = "기존 파일 불러오기"

print(f"Manifest: {manifest_action}")
print(f"Path: {MANIFEST_PATH}")

## Checks

원본 MP4 개수와 Manifest 행 수, 중복 ID, 영상 읽기 상태를 확인합니다. 문제가 있으면 다음 단계로 넘어가지 않고 여기서 원인을 확인합니다.

In [ ]:
video_count = len(list(VIDEO_DIR.glob("*.mp4")))
if video_count == 0:
    raise FileNotFoundError(f"MP4 파일이 없습니다: {VIDEO_DIR}")
if len(manifest) != video_count:
    raise ValueError(f"영상 {video_count}개와 Manifest {len(manifest)}행이 일치하지 않습니다.")
if manifest["video_id"].duplicated().any():
    raise ValueError("중복 video_id가 있습니다.")

diagnosis_summary = pd.Series(
    {
        "video_files": video_count,
        "manifest_rows": len(manifest),
        "unique_people": manifest["person_id"].nunique(),
        "read_ok": int((manifest["read_status"] == "ok").sum()),
        "read_error": int((manifest["read_status"] != "ok").sum()),
    },
    name="count",
)
diagnosis_summary.to_frame()

### 3. Manifest 일부 확인

출력이 너무 길어지지 않도록 앞의 5개 행만 확인합니다.

In [ ]:
manifest.head(5)

## U2. 영상 프레임과 시간 기준 확인

여기서는 Pose와 원판 분석이 함께 사용할 기준을 눈으로 확인합니다. 영상의 실제 FPS를 사용해 `timestamp_sec = frame_index / fps`로 계산하며, generator가 한 번에 한 프레임만 전달합니다. 표에는 이미지 전체가 아니라 작은 프레임 메타데이터만 저장합니다.

### 4. 실험값 설정

`VIDEO_ID`로 영상을 바꾸고, `PREVIEW_FRAME_INDEX`로 보고 싶은 프레임을 선택하세요. `SAMPLE_EVERY_N_FRAMES`는 시간표에 표시할 간격이며 영상 처리 간격을 뜻하지 않습니다.

In [ ]:
VIDEO_ID = B00_VIDEO_ID
PREVIEW_FRAME_INDEX = 0
SAMPLE_EVERY_N_FRAMES = 60

### 5. 프레임을 순서대로 읽고 기준 검사

모든 프레임을 `list`로 보관하지 않습니다. 현재 프레임에서 인덱스, 시간, 크기만 확인한 뒤 다음 프레임으로 넘어갑니다.

In [ ]:
if SAMPLE_EVERY_N_FRAMES <= 0:
    raise ValueError("SAMPLE_EVERY_N_FRAMES는 1 이상이어야 합니다.")

selected_rows = manifest.loc[manifest["video_id"] == VIDEO_ID]
if selected_rows.empty:
    available_ids = ", ".join(manifest["video_id"].head(5))
    raise ValueError(f"Manifest에 VIDEO_ID={VIDEO_ID!r}가 없습니다. 예시: {available_ids}")

selected_video = selected_rows.iloc[0]
video_path = Path(selected_video["path"])
expected_frame_count = int(selected_video["frame_count"])
video_properties = read_video_properties(video_path)

frame_samples = []
decoded_frame_count = 0
first_frame_index = None
last_frame_index = None
last_timestamp_sec = None
frame_shape = None

for frame_index, timestamp_sec, frame in iter_video_frames(
    video_path,
    expected_frame_count=expected_frame_count,
):
    if first_frame_index is None:
        first_frame_index = frame_index
        frame_shape = frame.shape
    last_frame_index = frame_index
    last_timestamp_sec = timestamp_sec
    decoded_frame_count += 1

    if frame_index % SAMPLE_EVERY_N_FRAMES == 0:
        frame_samples.append(
            {
                "frame_index": frame_index,
                "timestamp_sec": round(timestamp_sec, 6),
                "frame_shape": str(frame.shape),
            }
        )

if frame_samples[-1]["frame_index"] != last_frame_index:
    frame_samples.append(
        {
            "frame_index": last_frame_index,
            "timestamp_sec": round(last_timestamp_sec, 6),
            "frame_shape": str(frame_shape),
        }
    )

### 6. 영상 입력 계약 요약

`PASS`가 나오면 Manifest와 실제 디코딩 결과가 같은 프레임·시간 기준을 사용한다는 뜻입니다.

In [ ]:
contract_checks = {
    "starts_at_zero": first_frame_index == 0,
    "count_matches_manifest": decoded_frame_count == expected_frame_count,
    "last_index_matches_count": last_frame_index == decoded_frame_count - 1,
    "fps_matches_manifest": abs(
        float(video_properties["fps"]) - float(selected_video["fps"])
    ) < 1e-6,
    "shape_matches_manifest": frame_shape
    == (int(selected_video["height"]), int(selected_video["width"]), 3),
}
contract_status = "PASS" if all(contract_checks.values()) else "FAIL"

video_contract_summary = pd.Series(
    {
        "video_id": VIDEO_ID,
        "FPS": video_properties["fps"],
        "프레임 범위": f"{first_frame_index} ~ {last_frame_index}",
        "Manifest 프레임 수": expected_frame_count,
        "실제 디코딩 프레임 수": decoded_frame_count,
        "프레임 크기": str(frame_shape),
        "색상 순서": "BGR",
        "읽기 방식": "generator streaming",
        "계약 검사": contract_status,
    },
    name="value",
)
video_contract_summary.to_frame()

In [ ]:
failed_checks = [name for name, passed in contract_checks.items() if not passed]
if failed_checks:
    raise RuntimeError(f"영상 입력 계약 실패: {', '.join(failed_checks)}")

validated_video_id = VIDEO_ID
validated_video_path = video_path

pd.Series(contract_checks, name="passed").to_frame()

### 7. 프레임과 시간 표본 확인

첫 프레임, 설정한 간격의 프레임과 마지막 프레임을 확인합니다. 각 시간은 실제 FPS를 사용해 계산됩니다.

In [ ]:
pd.DataFrame(frame_samples)

### 8. 선택 프레임 확인

OpenCV는 BGR 순서로 읽으므로 화면에 표시할 때만 RGB로 바꿉니다. 원본 프레임의 좌우 방향은 변경하지 않습니다.

In [ ]:
if VIDEO_ID != validated_video_id:
    raise RuntimeError(
        "VIDEO_ID를 변경했다면 5~6단계 셀을 다시 실행해 영상 계약을 확인하세요."
    )

preview_frame = read_video_frame(validated_video_path, PREVIEW_FRAME_INDEX)
preview_rgb = cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 8))
plt.imshow(preview_rgb)
plt.title(f"{validated_video_id} | frame {PREVIEW_FRAME_INDEX}")
plt.axis("off")
plt.show()

### 9. 관찰 기록

- 확인한 `VIDEO_ID`:
- 변경한 실험값:
- 프레임·시간 기준에서 발견한 점:
- 좌우 방향 또는 영상 읽기 문제:

## Next Steps

다음 작업에서는 Manifest 분포와 실패 영상을 확인한 뒤 `has_plate`와 `camera_motion`을 기록합니다. 수동 기록 이후에는 `REBUILD_MANIFEST=False`를 유지합니다. Pose와 원판 결과도 이 U2의 `video_id`, `frame_index`, `timestamp_sec` 기준을 그대로 사용합니다.